<a href="https://colab.research.google.com/github/fawadwazir/flyrank-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fawadwazir/flyrank-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Ranking / scoring, built on top of a classification sub-model.**

My lane's real question is "which pages should a reviewer look at first" — that's a "which ones first?" question, which maps to **ranking/scoring** (target = a priority score, metric = precision@K), not plain classification.

Under the hood, the pipeline still trains a **binary classifier** (`is_declining_label`, decline vs not) because a probability is the easiest number to sort by. But the classifier's output only matters once it's turned into an ordered queue a reviewer works down with limited capacity. So: classification underneath, ranking/scoring on top.

It is not clustering (I'm not looking for unlabeled groups) and it is not plain classification either (a flat "declining: yes/no" list still leaves a reviewer with no idea which page to open first).

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/fawadwazir/flyrank-Internship"
REPO_DIR = "flyrank-Internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

flagged = (df["trend_direction"] == "down").sum()
print(f"Total pages: {len(df):,}")
print(f"Pages flagged 'down': {flagged:,} ({flagged/len(df):.1%})")

Total pages: 30,000
Pages flagged 'down': 16,262 (54.2%)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [2]:
print(df["trend_direction"].value_counts())
print()

label_rate = (df["trend_direction"] == "down").mean()
print(f"is_declining_label positive rate: {label_rate:.1%}")
print()

sample = df[["content_id", "impressions_last_30d", "impressions_prev_30d", "trend_pct", "trend_direction"]].head(5)
print(sample)

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

is_declining_label positive rate: 54.2%

             content_id  impressions_last_30d  impressions_prev_30d  \
0  content_304f48230142                   578                   987   
1  content_a1fb4e703a9e                  2501                  5915   
2  content_9aa793d4d895                  2382                  6089   
3  content_331d6c4de07b                  3626                  4206   
4  content_d99b7a2d90ca                  4211                  6452   

   trend_pct trend_direction  
0      -41.4            down  
1      -57.7            down  
2      -60.9            down  
3      -13.8          stable  
4      -34.7            down  


**Target: `is_declining_label`** = 1 when `trend_direction == "down"`, else 0. `trend_direction` comes from `trend_pct`, which compares `impressions_last_30d` against `impressions_prev_30d` (>20% drop = "down").

This is a **proxy, not a clean future outcome** — it's computed from the current 90-day window, not from something that happens after a decision point. Because `trend_direction` and `trend_pct` are literally how the label is built, they must never be used as model features (that's the leakage trap the data dictionary warns about).

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50.** Of the top 50 pages the ranking puts first, how many actually carry the positive label? This matches the real decision — a reviewer doesn't review the whole list, they review a fixed number of pages per cycle, so what happens at the *top* of the ranking is what actually gets acted on.

Plain accuracy would be a weak choice: the positive rate is 54.2%, close to a coin flip, so a mediocre ranking could still look "accurate" while putting useless pages at the top. Precision@50 can't hide behind that.

In [3]:
label_rate = (df["trend_direction"] == "down").mean()
print(f"Base rate (positive label share): {label_rate:.1%}")
print("-> a ranking that's no better than random should score close to this at any K.")
print()

with open("outputs/model_report.md") as f:
    report_lines = f.readlines()

start = next(i for i, l in enumerate(report_lines) if l.startswith("| Model |"))
for line in report_lines[start:start + 5]:
    print(line.rstrip())

Base rate (positive label share): 54.2%
-> a ranking that's no better than random should score close to this at any K.

| Model | ROC AUC | Avg precision | Precision@50 | Recall | F1 |
|---|---:|---:|---:|---:|---:|
| decision_tree | 0.742 | 0.575 | 0.540 | 0.716 | 0.634 |
| logistic_regression | 0.700 | 0.522 | 0.400 | 0.567 | 0.566 |
| random_forest | 0.750 | 0.618 | 0.740 | 0.744 | 0.640 |


**Metric: Precision@50.** Of the top 50 pages the ranking puts first, how many actually carry the positive label? This matches the real decision — a reviewer doesn't review the whole list, they review a fixed number of pages per cycle, so what happens at the *top* of the ranking is what actually gets acted on.

Plain accuracy would be a weak choice: the positive rate is 54.2%, close to a coin flip, so a mediocre ranking could still look "accurate" while putting useless pages at the top. Precision@50 can't hide behind that.

**Metric: Precision@50.** Of the top 50 pages the ranking puts first, how many actually carry the positive label? This matches the real decision — a reviewer doesn't review the whole list, they review a fixed number of pages per cycle, so what happens at the *top* of the ranking is what actually gets acted on.

Plain accuracy would be a weak choice: the positive rate is 54.2%, close to a coin flip, so a mediocre ranking could still look "accurate" while putting useless pages at the top. Precision@50 can't hide behind that.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one pseudonymized content item (a single page), aggregated over a trailing 90-day window.** `content_id` is unique per row (30,000 rows, 30,000 unique ids), and `client_id` groups pages by which of the 32 pseudonymized clients they belong to — useful for a client-holdout split later, never as a feature itself.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"content_id unique: {df['content_id'].is_unique}")
print(f"distinct client_id values: {df['client_id'].nunique()}")
print()

cols = ["content_id", "client_id", "impressions_90d", "clicks_90d", "ctr",
        "avg_position", "days_since_last_update", "word_count",
        "trend_direction", "trend_pct"]
df[cols].head(5)

Shape: 30,000 rows x 44 columns
content_id unique: True
distinct client_id values: 32



,content_id,client_id,impressions_90d,clicks_90d,ctr,avg_position,days_since_last_update,word_count,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,3803,29,0.76,10.6,20,3221.0,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,15320,7,0.05,20.3,25,2481.0,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,0.09,36.5,20,3515.0,down,-60.9
3,content_331d6c4de07b,client_19581e27de,11751,58,0.49,6.2,22,NaN,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,0.13,44.0,14,2803.0,down,-34.7


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule already exists to compare against — the baseline_rules score is a hand-weighted formula over four signals (visibility, freshness risk, position opportunity, depth gap). It's not a strawman; it's real, and it's beatable:

| Method | ROC AUC | Precision@50 |
|---|---:|---:|
| baseline rules | 0.627 | 0.240 |
| random forest | 0.750 | 0.740 |

That's roughly a **3x jump in Precision@50** on the exact same 30,000 rows. The signals genuinely interact rather than add up cleanly — a page can be stale AND low-traffic (not worth touching), or stale AND high-traffic (worth touching a lot). A linear weighted sum treats those cases the same way; a model that learns interactions doesn't have to.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
stale = df[df["days_since_last_update"] >= 180]

low_traffic_stale = stale[stale["impressions_90d"] < 100]
high_traffic_stale = stale[stale["impressions_90d"] >= 3000]

print(f"Stale pages (>=180 days since update): {len(stale):,}")
print(f"  low-traffic (<100 impressions/90d): {len(low_traffic_stale):,}")
print(f"  high-traffic (>=3000 impressions/90d): {len(high_traffic_stale):,}")

Stale pages (>=180 days since update): 174
  low-traffic (<100 impressions/90d): 139
  high-traffic (>=3000 impressions/90d): 9


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.